In [ ]:
""" LangExtract Product Information Extraction Example"""

from concurrent.futures import ThreadPoolExecutor
from datasets import load_dataset
from Modules.hyperparameters import get_hyperparameters
from Modules.load_db import get_dataframes
import json
import langextract as lx
import textwrap
import pandas as pd

In [ ]:
""" Constants """

DATASET = 'mave'
HYPER = get_hyperparameters()
# Set to 0 to process all entries
STARTING_POINT = HYPER[DATASET]['starting_point']
# Number of threads for parallel processing
MAX_WORKERS = HYPER[DATASET]['workers']
# Output file path
OUTPUT_PATH = HYPER[DATASET]['output_path']

PROMPT = textwrap.dedent(HYPER['prompts']['system']['mave_task'])

In [ ]:
""" Functions: get_lex_Example, get_examples, check_last_jsonl_id, get_safe_texts """


def get_lex_Example(example_item: dict) -> lx.data.ExampleData:
    return lx.data.ExampleData(
        text=example_item['text'],
        extractions=[
            lx.data.Extraction(
                extraction_class=example_item['class'],
                extraction_text=example_item['ext_text'],
                attributes=example_item['attributes'],
            ),
        ],
    )


def get_examples() -> list[lx.data.ExampleData]:
    """ Generate example data for LangExtract. """

    examples_data = [
        {'id': 1,
         'text': 'Camiseta PoloTech masculina de algodão, cor azul marinho, disponível nos tamanhos M, G e GG.',
         'class': 'product',
         'ext_text': 'Camiseta PoloTech masculina',
            'attributes': {
                'brand': 'PoloTech',
                'category': 'camiseta',
                'material': 'algodão',
                'color': 'azul marinho',
                'sizes': ['M', 'G', 'GG']
            }
         },
        {'id': 2,
         'text': 'Tênis esportivo Nike Air Zoom branco, ideal para corrida.',
         'class': 'product',
         'ext_text': 'Tênis esportivo Nike Air Zoom branco',
            'attributes': {
                'brand': 'Nike',
                'category': 'tênis esportivo',
                'color': 'branco',
                'intended_use': 'corrida'
            }
         },
    ]
    examples = []
    for example_item in examples_data:
        examples.append(get_lex_Example(example_item))
    return examples


def check_last_jsonl_id() -> int:
    """ Check the last processed ID in the output file to resume processing. """
    try:
        with open(OUTPUT_PATH, "r") as f:
            lines = f.readlines()
            if lines:
                last_record = json.loads(lines[-2])
                starting_point = last_record.get("id", -1) + 1
                if HYPER['mave']['verbose']:
                    print(f"Resuming from ID: {starting_point}")
                return starting_point
    except FileNotFoundError:
        return STARTING_POINT
    return STARTING_POINT

In [ ]:
""" Second Round of Constants """

EXAMPLES = get_examples()
STARTING_POINT = check_last_jsonl_id()
SAFE_TEXTS = get_dataframes('mave')['test']['text'][STARTING_POINT:]

In [ ]:
""" Extraction Function """


def format_output(idx: int, extraction_result: dict, text: str) -> dict:
    """ Format the extraction result into the desired output structure. """
    # Montar campos no mesmo formato do AE-110K
    attrs = extraction_result.extractions[0].attributes or {}
    attributes = list(attrs.keys())
    values = list(attrs.values())
    values_indices = []
    values_text = " | ".join(map(str, values))
    attributes_values = " | ".join(
        f"attribute: {k}, value: {v}" for k, v in attrs.items()
    )
    # json_answer = str(attrs)  # igual ao df (aspas simples)
    json_answer = json.dumps(attrs, ensure_ascii=False)

    record = {
        "id": idx,
        "text": text,
        "attributes": attributes,
        "values": values,
        "values_indices": values_indices,
        "values_text": values_text,
        "attributes_values": attributes_values,
        "json_answer": json_answer,
    }
    return record


def extract_text(idx, text):
    try:
        result = lx.extract(
            text_or_documents=text,
            prompt_description=PROMPT,
            examples=EXAMPLES,
            model_id=HYPER['mave']['model_id'],
            model_url=HYPER['mave']['model_url'],
            fence_output=False,
            # max_workers=10,
            use_schema_constraints=False,
            language_model_params={"timeout": 900}
        )

        if not result.extractions:
            return idx, None
        record = format_output(idx, result, text)
        return idx, record
    except Exception as err:
        return idx, {"error": str(err)}


def save_record_to_file(idx: int, record: dict, file) -> None:
    """ Save a single record to the output file. """
    if record and "error" not in record:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")
        print(f'[{idx}] OK')
    else:
        print(f'[{idx}] ERROR: {record}')

In [ ]:
""" Testing """


def testing():
    for i in range(5):
        idx, record = extract_text(i, texts[i])
        with open(OUTPUT_PATH, "a") as f:
            save_record_to_file(idx, record, f)


# testing()

In [ ]:
""" Paralelização """


def parallel_extraction(workers: int = MAX_WORKERS, output_path: str = OUTPUT_PATH):
    """ Perform parallel extraction and save results to output file. """
    print(f'Saving results to {output_path} using {workers} workers.')
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor, open(output_path, "a") as f:
        for i, record in executor.map(lambda args: extract_text(*args), enumerate(SAFE_TEXTS, start=STARTING_POINT)):
            save_record_to_file(i, record, f)

In [ ]:
parallel_extraction(HYPER['mave']['workers'], HYPER['mave']['output_path'])